# E01 Readiness and Launch Notebook

This notebook turns the E01 operator checklist into a concrete workflow.

## Purpose

E01 is the **target granularity experiment**. The goal is to compare:

- `emotion`
- `coarse_affect`
- `binary_valence_like`

under the fixed core setup:

- model: `ridge`
- primary CV: `within_subject_loso_session`

## Recommended execution policy

Use the **registry path**, not the workbook path, as the source of truth for execution.

## Workflow

1. Freeze the exact E01 definition
2. Run a dry-run of E01
3. Audit dataset sufficiency by target / subject / session
4. Verify index and feature-cache integrity
5. Run one smoke variant
6. Launch the full E01 primary batch
7. Review outputs and make the D01 decision
8. Decide whether to add the secondary transfer check


## Notes

- The notebook assumes you run it from the **repository root**.
- Command cells use `python -m uv run ...` to match your project style.
- The audit cells are designed to be helpful even before a full run.
- Some cells are intentionally conservative and may need small path edits if your local environment differs.


In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys
from typing import Iterable

import pandas as pd

REPO_ROOT = Path("..").resolve()
INDEX_CSV = REPO_ROOT / "Data/processed/dataset_index.csv"
DATA_ROOT = REPO_ROOT / "Data"
CACHE_DIR = REPO_ROOT / "Data/processed/feature_cache"
REGISTRY = REPO_ROOT / "configs/decision_support_registry.json"
OUTPUT_ROOT = REPO_ROOT / "outputs/artifacts/decision_support"
EXPERIMENT_ID = "E01"

print("Repo root:", REPO_ROOT)
print("Index exists:", INDEX_CSV.exists(), INDEX_CSV)
print("Data root exists:", DATA_ROOT.exists(), DATA_ROOT)
print("Cache dir exists:", CACHE_DIR.exists(), CACHE_DIR)
print("Registry exists:", REGISTRY.exists(), REGISTRY)
print("Output root:", OUTPUT_ROOT)


Repo root: D:\Khaled\Projects\VScode\Thesis
Index exists: True D:\Khaled\Projects\VScode\Thesis\Data\processed\dataset_index.csv
Data root exists: True D:\Khaled\Projects\VScode\Thesis\Data
Cache dir exists: True D:\Khaled\Projects\VScode\Thesis\Data\processed\feature_cache
Registry exists: True D:\Khaled\Projects\VScode\Thesis\configs\decision_support_registry.json
Output root: D:\Khaled\Projects\VScode\Thesis\outputs\artifacts\decision_support


## 1. Freeze the exact E01 definition

This cell checks the registry entry so you confirm E01 means exactly what you intend before running anything.


In [2]:
registry_data = json.loads(REGISTRY.read_text(encoding="utf-8"))
registry_data.keys()


dict_keys(['schema_version', 'description', 'experiments'])

In [3]:
# Try to locate the E01 entry in a reasonably flexible way.
e01_entry = None

if isinstance(registry_data, dict):
    if "experiments" in registry_data and isinstance(registry_data["experiments"], list):
        for item in registry_data["experiments"]:
            if isinstance(item, dict) and item.get("experiment_id") == EXPERIMENT_ID:
                e01_entry = item
                break
    elif EXPERIMENT_ID in registry_data:
        e01_entry = registry_data[EXPERIMENT_ID]

if e01_entry is None:
    print("Could not automatically locate E01 in the registry. Inspect the printed structure manually.")
else:
    from pprint import pprint
    pprint(e01_entry)


{'blocked_reasons': [],
 'dataset_scope': 'Internal BAS2 repeated-session index.',
 'decision_id': 'D01',
 'executable_now': True,
 'execution_status': 'executable',
 'experiment_id': 'E01',
 'fixed_controls': 'ridge model, within_subject_loso_session split, seed=42, '
                   'no task/modality filter.',
 'manipulated_factor': 'Target definition',
 'models': ['ridge'],
 'notes': 'Compares emotion, coarse affect, and binary valence-like targets '
          'under matched split/model controls.',
 'output_dir': 'outputs/artifacts/decision_support/E01',
 'primary_metric': 'balanced_accuracy',
 'secondary_metrics': ['macro_f1', 'accuracy'],
 'split_logic': 'within_subject_loso_session, per subject.',
 'stage': 'Stage 1 - Target lock',
 'target_definition': 'coarse_affect vs emotion vs binary valence-like.',
 'title': 'Target granularity experiment',
 'variant_templates': [{'expand': {'subject': 'subjects'},
                        'params': {'cv': 'within_subject_loso_session',
 

## 2. Dry-run E01

Expected outcome:

- E01 resolves successfully
- it expands into concrete variants
- the variants use:
  - `ridge`
  - `within_subject_loso_session`
  - one of the three target definitions

This cell prints the command first, then runs it.


In [4]:
dry_run_cmd = [
    sys.executable, "-m", "uv", "run",
    "thesisml-run-decision-support",
    "--registry", str(REGISTRY),
    "--experiment-id", EXPERIMENT_ID,
    "--index-csv", str(INDEX_CSV),
    "--data-root", str(DATA_ROOT),
    "--cache-dir", str(CACHE_DIR),
    "--output-root", str(OUTPUT_ROOT),
    "--dry-run",
]
print(" ".join(f'"{part}"' if " " in part else part for part in dry_run_cmd))


d:\Khaled\Projects\VScode\Thesis\.venv-gpu\Scripts\python.exe -m uv run thesisml-run-decision-support --registry D:\Khaled\Projects\VScode\Thesis\configs\decision_support_registry.json --experiment-id E01 --index-csv D:\Khaled\Projects\VScode\Thesis\Data\processed\dataset_index.csv --data-root D:\Khaled\Projects\VScode\Thesis\Data --cache-dir D:\Khaled\Projects\VScode\Thesis\Data\processed\feature_cache --output-root D:\Khaled\Projects\VScode\Thesis\outputs\artifacts\decision_support --dry-run


In [5]:
# Uncomment to actually execute the dry-run.
# result = subprocess.run(dry_run_cmd, text=True, capture_output=True)
# print("Return code:", result.returncode)
# print(result.stdout)
# print(result.stderr)


## 3. Dataset sufficiency audit

This is the most important scientific gate before the full E01 launch.

The audit below checks, for each candidate target:
- overall class counts
- per-subject class counts
- per-subject/per-session class coverage

It helps answer:
- is `emotion` too sparse?
- is `binary_valence_like` too easy but too reductive?
- is `coarse_affect` the best compromise?


In [6]:
df = pd.read_csv(INDEX_CSV)
print(df.shape)
print(df.columns.tolist())
df.head()


(3402, 17)
['sample_id', 'subject', 'session', 'bas', 'run', 'task', 'emotion', 'coarse_affect', 'modality', 'beta_index', 'beta_file', 'beta_path', 'mask_path', 'regressor_label', 'raw_label', 'subject_session', 'subject_session_bas']


,sample_id,subject,session,bas,run,task,emotion,coarse_affect,modality,beta_index,beta_file,beta_path,mask_path,regressor_label,raw_label,subject_session,subject_session_bas
0,sub-001_ses-01_BAS2_run-1_beta-1,sub-001,ses-01,BAS2,1,passive,anger,negative,audio,1,beta_0001.nii,sub-001/ses-01/BAS2/beta_0001.nii,sub-001/ses-01/BAS2/mask.nii,run-1_passive_anger_audio,run-1_passive_anger_audio,sub-001_ses-01,sub-001_ses-01_BAS2
1,sub-001_ses-01_BAS2_run-1_beta-2,sub-001,ses-01,BAS2,1,passive,anger,negative,video,2,beta_0002.nii,sub-001/ses-01/BAS2/beta_0002.nii,sub-001/ses-01/BAS2/mask.nii,run-1_passive_anger_video,run-1_passive_anger_video,sub-001_ses-01,sub-001_ses-01_BAS2
2,sub-001_ses-01_BAS2_run-1_beta-3,sub-001,ses-01,BAS2,1,passive,anger,negative,audiovisual,3,beta_0003.nii,sub-001/ses-01/BAS2/beta_0003.nii,sub-001/ses-01/BAS2/mask.nii,run-1_passive_anger_audiovisual,run-1_passive_anger_audiovisual,sub-001_ses-01,sub-001_ses-01_BAS2
3,sub-001_ses-01_BAS2_run-1_beta-4,sub-001,ses-01,BAS2,1,passive,anxiety,negative,audio,4,beta_0004.nii,sub-001/ses-01/BAS2/beta_0004.nii,sub-001/ses-01/BAS2/mask.nii,run-1_passive_anxiety_audio,run-1_passive_anxiety_audio,sub-001_ses-01,sub-001_ses-01_BAS2
4,sub-001_ses-01_BAS2_run-1_beta-5,sub-001,ses-01,BAS2,1,passive,anxiety,negative,video,5,beta_0005.nii,sub-001/ses-01/BAS2/beta_0005.nii,sub-001/ses-01/BAS2/mask.nii,run-1_passive_anxiety_video,run-1_passive_anxiety_video,sub-001_ses-01,sub-001_ses-01_BAS2


In [7]:
required_base_cols = ["subject", "session"]
missing_base = [c for c in required_base_cols if c not in df.columns]
if missing_base:
    raise ValueError(f"Missing required columns for the audit: {missing_base}")

candidate_targets = [c for c in ["emotion", "coarse_affect", "binary_valence_like"] if c in df.columns]
print("Candidate targets found:", candidate_targets)


Candidate targets found: ['emotion', 'coarse_affect']


In [8]:
def summarize_target(df: pd.DataFrame, target: str) -> dict:
    out = {}
    valid = df.dropna(subset=[target]).copy()
    out["n_rows"] = len(valid)
    out["n_subjects"] = valid["subject"].nunique()
    out["n_sessions"] = valid["session"].nunique()
    out["class_counts"] = valid[target].value_counts(dropna=False).to_dict()
    out["classes_per_subject"] = (
        valid.groupby("subject")[target]
        .nunique()
        .sort_values()
        .to_dict()
    )
    out["classes_per_subject_session"] = (
        valid.groupby(["subject", "session"])[target]
        .nunique()
        .sort_values()
        .to_dict()
    )
    return out

audit = {target: summarize_target(df, target) for target in candidate_targets}
audit


{'emotion': {'n_rows': 3402,
  'n_subjects': 2,
  'n_sessions': 24,
  'class_counts': {'anger': 378,
   'anxiety': 378,
   'disgust': 378,
   'happiness': 378,
   'interest': 378,
   'neutral': 378,
   'pride': 378,
   'relief': 378,
   'sadness': 378},
  'classes_per_subject': {'sub-001': 9, 'sub-002': 9},
  'classes_per_subject_session': {('sub-001', 'ses-01'): 9,
   ('sub-001', 'ses-02'): 9,
   ('sub-001', 'ses-03'): 9,
   ('sub-001', 'ses-04'): 9,
   ('sub-001', 'ses-05'): 9,
   ('sub-001', 'ses-06'): 9,
   ('sub-001', 'ses-07'): 9,
   ('sub-001', 'ses-08'): 9,
   ('sub-001', 'ses-09'): 9,
   ('sub-001', 'ses-10'): 9,
   ('sub-001', 'ses-11'): 9,
   ('sub-001', 'ses-12'): 9,
   ('sub-001', 'ses-13'): 9,
   ('sub-001', 'ses-14'): 9,
   ('sub-001', 'ses-15'): 9,
   ('sub-001', 'ses-16'): 9,
   ('sub-001', 'ses-17'): 9,
   ('sub-001', 'ses-18'): 9,
   ('sub-001', 'ses-19'): 9,
   ('sub-002', 'ses-01'): 9,
   ('sub-002', 'ses-02'): 9,
   ('sub-002', 'ses-03'): 9,
   ('sub-002', 'ses-05

In [9]:
for target, info in audit.items():
    print("=" * 90)
    print("TARGET:", target)
    print("Rows:", info["n_rows"])
    print("Subjects:", info["n_subjects"])
    print("Sessions:", info["n_sessions"])
    print("Class counts:")
    print(pd.Series(info["class_counts"]).sort_values(ascending=False))
    print()
    cps = pd.Series(info["classes_per_subject"]).sort_values()
    print("Number of unique classes per subject:")
    print(cps)
    print()
    cpss = pd.Series(info["classes_per_subject_session"]).sort_values()
    print("Number of unique classes per subject-session (smallest first):")
    print(cpss.head(20))
    print()


TARGET: emotion
Rows: 3402
Subjects: 2
Sessions: 24
Class counts:
anger        378
anxiety      378
disgust      378
happiness    378
interest     378
neutral      378
pride        378
relief       378
sadness      378
dtype: int64

Number of unique classes per subject:
sub-001    9
sub-002    9
dtype: int64

Number of unique classes per subject-session (smallest first):
sub-001  ses-01    9
         ses-02    9
         ses-03    9
         ses-04    9
         ses-05    9
         ses-06    9
         ses-07    9
         ses-08    9
         ses-09    9
         ses-10    9
         ses-11    9
         ses-12    9
         ses-13    9
         ses-14    9
         ses-15    9
         ses-16    9
         ses-17    9
         ses-18    9
         ses-19    9
sub-002  ses-01    9
dtype: int64

TARGET: coarse_affect
Rows: 3402
Subjects: 2
Sessions: 24
Class counts:
negative    1512
positive    1512
neutral      378
dtype: int64

Number of unique classes per subject:
sub-001    3
sub-

### LOSO session viability check

For each target and subject, this estimates whether session-level leave-one-session-out looks viable in a basic sense.

This is not a full proof of modeling viability, but it catches obvious cases where held-out sessions may be missing class coverage.


In [10]:
def loso_session_viability(df: pd.DataFrame, target: str) -> pd.DataFrame:
    rows = []
    valid = df.dropna(subset=[target]).copy()
    for subject, sdf in valid.groupby("subject"):
        all_classes = set(sdf[target].dropna().unique().tolist())
        sessions = sorted(sdf["session"].dropna().unique().tolist())
        for heldout_session in sessions:
            train = sdf[sdf["session"] != heldout_session]
            test = sdf[sdf["session"] == heldout_session]
            train_classes = set(train[target].dropna().unique().tolist())
            test_classes = set(test[target].dropna().unique().tolist())
            rows.append({
                "target": target,
                "subject": subject,
                "heldout_session": heldout_session,
                "n_train": len(train),
                "n_test": len(test),
                "n_all_classes": len(all_classes),
                "n_train_classes": len(train_classes),
                "n_test_classes": len(test_classes),
                "missing_from_train": sorted(test_classes - train_classes),
                "missing_from_test": sorted(all_classes - test_classes),
                "train_has_all_global_classes": train_classes == all_classes,
            })
    return pd.DataFrame(rows)

viability_frames = []
for target in candidate_targets:
    viability_frames.append(loso_session_viability(df, target))

viability = pd.concat(viability_frames, ignore_index=True) if viability_frames else pd.DataFrame()
viability.head()


,target,subject,heldout_session,n_train,n_test,n_all_classes,n_train_classes,n_test_classes,missing_from_train,missing_from_test,train_has_all_global_classes
0,emotion,sub-001,ses-01,1458,81,9,9,9,[],[],True
1,emotion,sub-001,ses-02,1458,81,9,9,9,[],[],True
2,emotion,sub-001,ses-03,1458,81,9,9,9,[],[],True
3,emotion,sub-001,ses-04,1458,81,9,9,9,[],[],True
4,emotion,sub-001,ses-05,1458,81,9,9,9,[],[],True


In [11]:
if not viability.empty:
    display_cols = [
        "target", "subject", "heldout_session",
        "n_train", "n_test",
        "n_all_classes", "n_train_classes", "n_test_classes",
        "train_has_all_global_classes", "missing_from_train"
    ]
    problems = viability[
        (~viability["train_has_all_global_classes"]) |
        (viability["n_test"] == 0) |
        (viability["n_train"] == 0)
    ][display_cols].copy()
    print("Potential LOSO viability problems:")
    display(problems if len(problems) else pd.DataFrame(columns=display_cols))


Potential LOSO viability problems:


,target,subject,heldout_session,n_train,n_test,n_all_classes,n_train_classes,n_test_classes,train_has_all_global_classes,missing_from_train


## 4. Feature-cache and index integrity checks

These checks are meant to catch avoidable I/O and cache issues before a real run.


In [12]:
print("Index exists:", INDEX_CSV.exists())
print("Cache dir exists:", CACHE_DIR.exists())
if CACHE_DIR.exists():
    npz_files = list(CACHE_DIR.rglob("*.npz"))
    print("Number of .npz cache files found:", len(npz_files))
    print("First few cache files:")
    for p in npz_files[:10]:
        print(" -", p)


Index exists: True
Cache dir exists: True
Number of .npz cache files found: 49
First few cache files:
 - D:\Khaled\Projects\VScode\Thesis\Data\processed\feature_cache\sub-002\ses-01\BAS2.npz
 - D:\Khaled\Projects\VScode\Thesis\Data\processed\feature_cache\sub-002\ses-02\BAS2.npz
 - D:\Khaled\Projects\VScode\Thesis\Data\processed\feature_cache\sub-002\ses-03\BAS2.npz
 - D:\Khaled\Projects\VScode\Thesis\Data\processed\feature_cache\sub-002\ses-05\BAS2.npz
 - D:\Khaled\Projects\VScode\Thesis\Data\processed\feature_cache\sub-002\ses-06\BAS2.npz
 - D:\Khaled\Projects\VScode\Thesis\Data\processed\feature_cache\sub-002\ses-07\BAS2.npz
 - D:\Khaled\Projects\VScode\Thesis\Data\processed\feature_cache\sub-002\ses-08\BAS2.npz
 - D:\Khaled\Projects\VScode\Thesis\Data\processed\feature_cache\sub-002\ses-09\BAS2.npz
 - D:\Khaled\Projects\VScode\Thesis\Data\processed\feature_cache\sub-002\ses-10\BAS2.npz
 - D:\Khaled\Projects\VScode\Thesis\Data\processed\feature_cache\sub-002\ses-11\BAS2.npz


In [13]:
# Light sanity check for subjects in index
if "subject" in df.columns:
    print("Subjects:", sorted(df["subject"].dropna().unique().tolist())[:20])
    print("Number of subjects:", df["subject"].nunique())


Subjects: ['sub-001', 'sub-002']
Number of subjects: 2


## 5. Run one smoke variant first

Recommended first smoke case:
- one subject
- target = `coarse_affect`
- model = `ridge`
- CV = `within_subject_loso_session`

Edit `SMOKE_SUBJECT` if needed.


In [14]:
SMOKE_SUBJECT = sorted(df["subject"].dropna().unique().tolist())[0] if "subject" in df.columns and len(df) else "sub-001"
RUN_ID = f"exploratory_{SMOKE_SUBJECT}_ridge_coarse_affect_e01_smoke"

smoke_cmd = [
    "thesisml-run-experiment",
    "--index-csv", str(INDEX_CSV),
    "--data-root", str(DATA_ROOT),
    "--cache-dir", str(CACHE_DIR),
    "--target", "coarse_affect",
    "--model", "ridge",
    "--cv", "within_subject_loso_session",
    "--subject", str(SMOKE_SUBJECT),
    "--run-id", RUN_ID,
]
print(" ".join(f'"{part}"' if " " in part else part for part in smoke_cmd))


thesisml-run-experiment --index-csv D:\Khaled\Projects\VScode\Thesis\Data\processed\dataset_index.csv --data-root D:\Khaled\Projects\VScode\Thesis\Data --cache-dir D:\Khaled\Projects\VScode\Thesis\Data\processed\feature_cache --target coarse_affect --model ridge --cv within_subject_loso_session --subject sub-001 --run-id exploratory_sub-001_ridge_coarse_affect_e01_smoke


In [15]:
# Uncomment to actually execute the smoke run.
result = subprocess.run(smoke_cmd, text=True, capture_output=True)
print("Return code:", result.returncode)
print(result.stdout)
print(result.stderr)
#5 and half minuts

Return code: 1

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "d:\Khaled\Projects\VScode\Thesis\.venv-gpu\Scripts\thesisml-run-experiment.exe\__main__.py", line 10, in <module>
    sys.exit(main())
             ~~~~^^
  File "D:\Khaled\Projects\VScode\Thesis\src\Thesis_ML\experiments\run_experiment.py", line 1835, in main
    result = run_experiment(
        index_csv=Path(args.index_csv),
    ...<47 lines>...
        scheduled_compute_assignment=scheduled_assignment,
    )
  File "D:\Khaled\Projects\VScode\Thesis\src\Thesis_ML\experiments\run_experiment.py", line 843, in run_experiment
    run_mode = prepare_report_dir(
        report_dir,
    ...<2 lines>...
        resume=bool(resume),
    )
  File "D:\Khaled\Projects\VScode\Thesis\src\Thesis_ML\experiments\execution_policy.py", line 128, in prepare_report_dir
    raise FileExistsError(
    ...<3 lines>...
    )
FileExistsError: Run 

## 6. Launch the full primary E01 batch

Only do this after:
- the dry-run passes
- the sufficiency audit looks acceptable
- the smoke variant succeeds

This launches the full primary E01 batch from the registry path.


In [20]:
full_cmd = [
    sys.executable, "-m", "uv", "run",
    "thesisml-run-decision-support",
    "--registry", str(REGISTRY),
    "--experiment-id", EXPERIMENT_ID,
    "--index-csv", str(INDEX_CSV),
    "--data-root", str(DATA_ROOT),
    "--cache-dir", str(CACHE_DIR),
    "--output-root", str(OUTPUT_ROOT),
]
print(" ".join(f'"{part}"' if " " in part else part for part in full_cmd))


d:\Khaled\Projects\VScode\Thesis\.venv-gpu\Scripts\python.exe -m uv run thesisml-run-decision-support --registry D:\Khaled\Projects\VScode\Thesis\configs\decision_support_registry.json --experiment-id E01 --index-csv D:\Khaled\Projects\VScode\Thesis\Data\processed\dataset_index.csv --data-root D:\Khaled\Projects\VScode\Thesis\Data --cache-dir D:\Khaled\Projects\VScode\Thesis\Data\processed\feature_cache --output-root D:\Khaled\Projects\VScode\Thesis\outputs\artifacts\decision_support


In [21]:
import sys, os, platform
print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("Platform:", platform.platform())
print("Working directory:", os.getcwd())

Python executable: d:\Khaled\Projects\VScode\Thesis\.venv-gpu\Scripts\python.exe
Python version: 3.13.6 (tags/v3.13.6:4e66535, Aug  6 2025, 14:36:00) [MSC v.1944 64 bit (AMD64)]
Platform: Windows-10-10.0.19045-SP0
Working directory: d:\Khaled\Projects\VScode\Thesis\notebooks


In [27]:
import sys
!"{sys.executable}" -m ensurepip --upgrade

Looking in links: c:\Users\Khaled\AppData\Local\Temp\tmp5w4c77l3
Processing c:\users\khaled\appdata\local\temp\tmp5w4c77l3\pip-25.2-py3-none-any.whl


In [28]:
!pip install uv

In [29]:
%pip install uv

   ---------------------------------------- 0.0/24.2 MB ? eta -:--:--
   -------------- ------------------------- 8.9/24.2 MB 43.5 MB/s eta 0:00:01
   --------------------------- ------------ 16.8/24.2 MB 41.3 MB/s eta 0:00:01
   ------------------------------------- -- 22.8/24.2 MB 37.2 MB/s eta 0:00:01
   ---------------------------------------- 24.2/24.2 MB 30.7 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [30]:
# Uncomment to actually execute the full E01 batch.
result = subprocess.run(full_cmd, text=True, capture_output=True)
print("Return code:", result.returncode)
print(result.stdout)
print(result.stderr)


Return code: 0
Campaign complete
- campaign_id: 20260323_162326_159359
- campaign_root: D:\Khaled\Projects\VScode\Thesis\outputs\artifacts\decision_support\campaigns\20260323_162326_159359
- selected_experiments: E01
- status_counts: {'completed': 6}
- run_log_export: D:\Khaled\Projects\VScode\Thesis\outputs\artifacts\decision_support\campaigns\20260323_162326_159359\run_log_export.csv
- decision_summary: D:\Khaled\Projects\VScode\Thesis\outputs\artifacts\decision_support\campaigns\20260323_162326_159359\decision_support_summary.csv
- decision_report: D:\Khaled\Projects\VScode\Thesis\outputs\artifacts\decision_support\campaigns\20260323_162326_159359\decision_recommendations.md
- aggregation: D:\Khaled\Projects\VScode\Thesis\outputs\artifacts\decision_support\campaigns\20260323_162326_159359\result_aggregation.json
- summary_outputs_export: D:\Khaled\Projects\VScode\Thesis\outputs\artifacts\decision_support\campaigns\20260323_162326_159359\summary_outputs_export.csv
- manifest: D:\Khal

## 7. Post-run review guide

Do **not** choose the target only by one metric.

For each target, review:
- balanced accuracy
- macro-F1
- class-wise recall
- per-subject stability
- fold validity
- whether performance is inflated by easier target definition rather than better construct fit

Suggested decision rule:
- reject targets that are not data-sufficient
- prefer stable targets across subjects and sessions
- prefer targets that preserve thesis construct validity rather than merely maximizing raw ease of classification


## 8. Secondary transfer check decision

The workbook narrative mentions a secondary frozen-transfer check, while the executable registry may only implement the primary within-subject comparison.

After the primary E01 batch, explicitly decide one of these:
- E01 is only the within-subject target-lock experiment
- E01 requires a secondary cross-person transfer follow-up before D01 is locked


## 9. Write-back checklist

After E01 completes, update:
- Decision_Log for D01
- Run_Log
- Trial_Results
- optionally Data_Profile with class-count / sufficiency evidence

Record:
- winning target
- why it won
- why the alternatives did not
- any tradeoff between learnability and construct validity
- whether secondary transfer remains pending


## Optional next step

After this notebook, the most useful follow-up would be a small report notebook or script that aggregates the completed E01 artifacts into a target-by-subject comparison table and a D01 decision memo draft.
